# LIFE on Google Colab — PolitiFact++ (binary MF-vs-MR, LLaMA2-7B)

Faithful reproduction of the paper's setup: binary fake/real over the **LLM pair** (MF=fake, MR=real), with **LLaMA2-7B** as the reconstruction model. End-to-end: **convert → key-sentence extraction → concatenate → features → train**.

**Before you start:**
1. Set the Colab runtime to **GPU** — an **A100** is needed for LLaMA2-7B (Runtime → Change runtime type).
2. Upload the whole `LIFE` repo (including `dataset/data/`) to your Google Drive, e.g. `MyDrive/LIFE`.
3. Edit `PROJECT_DIR` in the path cell below if you put it somewhere else.
4. Step 3 uses the **ungated** `NousResearch/Llama-2-7b-hf` mirror by default — no HF token needed. (The official `meta-llama/Llama-2-7b-hf` is gated and requires an approved access request + token.)

Scope: **PolitiFact++** only (~229 LLM-pair articles: 97 fake + 132 real). VLPFN is excluded (its text has no punctuation, so sentence splitting cannot work). GossipCop++ is far heavier; try it only after this works.

In [11]:
# Confirm a GPU is attached
!nvidia-smi

Thu Jul  2 02:09:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             43W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [12]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
import os

# <-- change this if you uploaded the repo elsewhere
PROJECT_DIR = '/content/drive/MyDrive/LIFE'
os.chdir(PROJECT_DIR)

POLITIFACT_DIR = f'{PROJECT_DIR}/dataset/data/Fakenews-dataset-main/Fakenews-dataset-main/Dataset/PolitiFact++'

# Paper's binary task: LLM pair only (MF=fake, MR=real), reconstructed with LLaMA2-7B.
OUTPUT_BIN     = f'{PROJECT_DIR}/dataset/output_bin'        # MF_fake.jsonl + MR_true.jsonl
KEY_SENT       = f'{PROJECT_DIR}/dataset/keySentence/important_sentences_top10.jsonl'
BERT_CKPT      = f'{PROJECT_DIR}/dataset/bert_bin.pt'       # fresh extractor for MF-vs-MR
FEATURES_LLAMA = f'{PROJECT_DIR}/dataset/features_llama'
TRAIN_PATH     = f'{PROJECT_DIR}/dataset/train_bin.jsonl'
TEST_PATH      = f'{PROJECT_DIR}/dataset/test_bin.jsonl'

print('cwd:', os.getcwd())
print('PolitiFact++ found:', os.path.isdir(POLITIFACT_DIR))

cwd: /content/drive/MyDrive/LIFE
PolitiFact++ found: True


In [14]:
# Install dependencies.
# If the fastNLP import fails at the training step, pin a compatible version:
#   !pip install -q fastNLP==1.0.1
!pip install -q -r requirements.txt

In [15]:
# NLTK sentence tokenizer data. 'punkt' gives english.pickle (used by train.py);
# 'punkt_tab' is required by newer nltk's sent_tokenize (used by step 1).
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## HuggingFace login (optional)
Step 3 defaults to the **ungated** `NousResearch/Llama-2-7b-hf` mirror, so **no token is needed — you can skip this cell**. Only run it if you switch Step 3 to the official gated `meta-llama/Llama-2-7b-hf` (which also requires an approved access request).

In [ ]:
# LLaMA-2 is gated on HuggingFace. First accept the license at
# https://huggingface.co/meta-llama/Llama-2-7b-hf, then run this cell and paste an
# access token from https://huggingface.co/settings/tokens
# (or replace with: login(token="hf_xxx")).
from huggingface_hub import login
login()

## Step 0 — Convert PolitiFact++ to the binary LLM-pair JSONL
`--subset llm` emits only `MF_fake.jsonl` (97, fake) and `MR_true.jsonl` (132, real) — the paper's binary task. HF/HR (human-written) are not used.

In [ ]:
!python dataset/0_convert.py --input_dir "{POLITIFACT_DIR}" --output_dir "{OUTPUT_BIN}" --subset llm

## Step 1 — Key-sentence extraction (top-10)
Trains a **fresh** BERT fake/real classifier on MF-vs-MR (saved to `BERT_CKPT`), then keeps the **top-10** most impactful sentences per article (paper's k for PolitiFact++). This is the slowest step (a forward pass per sentence per article).

In [ ]:
!python dataset/1_keySentenceExtraction.py --data_dir "{OUTPUT_BIN}" --output_file "{KEY_SENT}" --top_k 10 --model_path "{BERT_CKPT}" --gpu 0

## Step 2 — Concatenate key sentences back into the data
Adds a `sentence` field to each record in `OUTPUT_BIN` by matching on `(id, label)`. **Overwrites the files in `OUTPUT_BIN` in place** — re-run Step 0 first if you need to reset them.

In [ ]:
!python dataset/2_concate.py --folder_path "{OUTPUT_BIN}" --important_sentences_file "{KEY_SENT}"

## Step 3 — Reconstruction probabilities with LLaMA2-7B
The paper's reconstruction model. A malicious prompt is prepended and LLaMA2-7B's per-token log-likelihoods over the key fragments form the "linguistic fingerprint" features. Loads in bfloat16 (~14 GB; needs the A100) and downloads ~13 GB on first run. Writes one feature JSONL per input file into `FEATURES_LLAMA`.

In [ ]:
# meta-llama/Llama-2-7b-hf is gated (needs Meta approval). NousResearch/Llama-2-7b-hf is an
# ungated mirror of the SAME weights/tokenizer — no token needed. Swap back to the official
# repo if/when your access request is approved.
!python dataset/3_gen_features_local.py --input_dir "{OUTPUT_BIN}" --output_dir "{FEATURES_LLAMA}" --model NousResearch/Llama-2-7b-hf --scorer llama --dtype bfloat16 --gpu 0

## Step 4A — train the classifier (released head: BMES tags + CRF + majority vote)
Splits `FEATURES_LLAMA` into train/test (seed-0, deterministic) and trains the released Transformer classifier for **50 epochs** on the binary MF-vs-MR task. This head diverges from the paper — it tags tokens with B/M/E/S labels, CRF-decodes them, and recovers the article label by majority vote — so it is the **A-side** of the head A/B. Step 4B below is the paper-faithful head. Paper target for PolitiFact++: **Acc 0.900 / F1 0.882**.

In [ ]:
!python LIFE_train/train.py \
  --split_dataset \
  --data_path "{FEATURES_LLAMA}" \
  --train_path "{TRAIN_PATH}" \
  --test_path "{TEST_PATH}" \
  --model Transformer \
  --num_train_epochs 50

## Step 4B — paper head (sigmoid + BCE, Eq 11–12)
Same CNN→Transformer trunk, but the head matches the paper: masked mean-pool → one sigmoid probability per article, trained with **binary cross-entropy** (fake=1, real=0) and evaluated directly at article level — no BMES tags, no CRF, no majority vote. Files: `LIFE_train/model_bce.py` + `LIFE_train/train_bce.py` (the originals are untouched and remain the A-side).

**The A/B is fair:** 4A and 4B consume the *same* `FEATURES_LLAMA` and the *same* seed-0 train/test split (`--split_dataset` here regenerates the identical split, so running 4A first is not required). The test set is only ~46 articles (≈2 accuracy points per article), so **re-run this cell with `--seed 1`, `2`, `3`, `4` and report mean ± std**. A-side references: 85.5/80.8 and 83.9/78.2; paper target 90.0/88.2. Checkpoint: `bce_en.pt`.

In [ ]:
!python LIFE_train/train_bce.py \
  --split_dataset \
  --data_path "{FEATURES_LLAMA}" \
  --train_path "{TRAIN_PATH}" \
  --test_path "{TEST_PATH}" \
  --num_train_epochs 50 \
  --seed 0

## Multiclass experiment — 4-class HF / HR / MF / MR (LLaMA2-7B, released CRF/BMES head)

A separate, exploratory run that classifies all **four** PolitiFact++ categories
(human-fake, human-true, gpt3.5-fake, gpt3.5-true) instead of the paper's binary MF-vs-MR.
It reuses the released CRF/BMES head via `LIFE_train/train_multi.py` (a copy of `train.py`
with `en_labels` set to the four classes; `model.py` / `dataloader.py` are imported unchanged).

This needs its own LLaMA2-7B features (the binary `FEATURES_LLAMA` only has MF/MR), so
Steps 0m–3m re-run the pipeline with `--subset all` into separate `*_multi` paths — nothing
above is overwritten. Reference: an earlier 4-class run with **gpt2** features scored ~51.7%;
this swaps in the LLaMA2-7B features.

In [ ]:
# --- 4-class (HF/HR/MF/MR) experiment paths (separate from the binary run above) ---
OUTPUT_MULTI     = f'{PROJECT_DIR}/dataset/output_multi'
KEY_SENT_MULTI   = f'{PROJECT_DIR}/dataset/keySentence/important_sentences_multi_top10.jsonl'
BERT_CKPT_MULTI  = f'{PROJECT_DIR}/dataset/bert_multi.pt'
FEATURES_MULTI   = f'{PROJECT_DIR}/dataset/features_llama_multi'
TRAIN_PATH_MULTI = f'{PROJECT_DIR}/dataset/train_multi.jsonl'
TEST_PATH_MULTI  = f'{PROJECT_DIR}/dataset/test_multi.jsonl'

### Step 0m — Convert all four categories
`--subset all` emits `HF_fake` / `MF_fake` / `HR_true` / `MR_true` JSONL (~520 articles).

In [ ]:
!python dataset/0_convert.py --input_dir "{POLITIFACT_DIR}" --output_dir "{OUTPUT_MULTI}" --subset all

### Step 1m — Key-sentence extraction (top-10)
Trains a fresh binary BERT (fake = HF+MF, true = HR+MR) and keeps the top-10 sentences per
article. Slowest step; now over ~520 articles.

In [ ]:
!python dataset/1_keySentenceExtraction.py --data_dir "{OUTPUT_MULTI}" --output_file "{KEY_SENT_MULTI}" --top_k 10 --model_path "{BERT_CKPT_MULTI}" --gpu 0

### Step 2m — Concatenate key sentences
Adds the `sentence` field to the records in `OUTPUT_MULTI` **in place** — re-run Step 0m to reset.

In [ ]:
!python dataset/2_concate.py --folder_path "{OUTPUT_MULTI}" --important_sentences_file "{KEY_SENT_MULTI}"

### Step 3m — LLaMA2-7B reconstruction features
Same as the binary Step 3 but over all four files → `FEATURES_MULTI`.

In [ ]:
!python dataset/3_gen_features_local.py --input_dir "{OUTPUT_MULTI}" --output_dir "{FEATURES_MULTI}" --model NousResearch/Llama-2-7b-hf --scorer llama --dtype bfloat16 --gpu 0

### Step 4m — Train the 4-class classifier
Released CRF/BMES head over 16 BMES tags (4 classes × B/M/E/S), recovered to a 4-class label
by sentence majority vote. Prints Accuracy / Macro-F1 / per-class precision-recall in the
class id order printed at startup. Writes `linear_multi_en.pt`.

In [ ]:
!python LIFE_train/train_multi.py --split_dataset --data_path "{FEATURES_MULTI}" --train_path "{TRAIN_PATH_MULTI}" --test_path "{TEST_PATH_MULTI}" --model Transformer --num_train_epochs 50

### Step 4c — Combined-by-veracity binary (fake = HF+MF, real = HR+MR)
A second side experiment that **pools the human dataset in**: the four categories are
collapsed to two veracity buckets by label suffix (`LIFE_train/train_combined.py`, a copy of
`train.py` with a `DataManager` subclass that maps `_fake`→fake / `_true`→true). Released
CRF/BMES head, sentence majority-vote eval. Per-class P/R prints in the class id order
(fake=0, true=1) printed at startup.

**Reuses `FEATURES_MULTI`** from the 4-class experiment (features are label-agnostic) — run
Steps 0m–3m first. `--split_dataset` regenerates the same seed-0 split as Step 4m, so the two
runs are directly comparable. Reference: a prior combined run scored ~85.2 Acc / 78.4 Macro-F1
(human-written fakes lack the LLM fingerprint, which drags fake recall down).

In [ ]:
!python LIFE_train/train_combined.py --split_dataset --data_path "{FEATURES_MULTI}" --train_path "{TRAIN_PATH_MULTI}" --test_path "{TEST_PATH_MULTI}" --model Transformer --num_train_epochs 50

## GPT-2 reconstruction model — binary MF-vs-MR (both heads)

Swaps the reconstruction LM from LLaMA2-7B to **GPT-2** on the paper's binary task, run through
both heads, to isolate the effect of the reconstruction model. No code changes:
`3_gen_features_local.py` scores with GPT-2 via `--scorer bbpe`, and `train.py` / `train_bce.py`
are already the binary released / BCE heads.

**Reuses the existing `OUTPUT_BIN`** (the key sentences from the LLaMA binary run's Step 2) — do
**not** re-run Steps 0–2, so the only difference vs the LLaMA run is the reconstruction LM here.
(If `OUTPUT_BIN` was reset since, re-run the binary Steps 0–2 first.)

LLaMA baselines: released head ~83.9–85.5 / ~78–80.8; BCE head 86.82 ± 2.157 / 84.48 ± 2.986
(seeds 0–20).

In [16]:
# GPT-2 binary paths (separate from the LLaMA binary run's FEATURES_LLAMA / TRAIN_PATH / TEST_PATH)
FEATURES_GPT2_BIN = f'{PROJECT_DIR}/dataset/features_gpt2_bin'
TRAIN_PATH_GPT2   = f'{PROJECT_DIR}/dataset/train_gpt2_bin.jsonl'
TEST_PATH_GPT2    = f'{PROJECT_DIR}/dataset/test_gpt2_bin.jsonl'

### Step 3g — GPT-2 reconstruction features
Scores the same `OUTPUT_BIN` articles with **GPT-2** (`--scorer bbpe`). GPT-2 is small — fast,
ungated, fits any GPU. Writes `MF_fake.jsonl` + `MR_true.jsonl` into `FEATURES_GPT2_BIN`.

In [17]:
!python dataset/3_gen_features_local.py --input_dir "{OUTPUT_BIN}" --output_dir "{FEATURES_GPT2_BIN}" --model gpt2 --scorer bbpe --gpu 0

device: cuda | model: gpt2 | dtype: bfloat16 | scorer: bbpe
config.json: 100% 665/665 [00:00<00:00, 2.78MB/s]
tokenizer_config.json: 100% 26.0/26.0 [00:00<00:00, 127kB/s]
vocab.json: 100% 1.04M/1.04M [00:00<00:00, 1.96MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 864kB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 2.97MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100% 548M/548M [00:03<00:00, 169MB/s]
Loading weights: 100% 148/148 [00:00<00:00, 2591.41it/s]
generation_config.json: 100% 124/124 [00:00<00:00, 676kB/s]
input file:/content/drive/MyDrive/LIFE/dataset/output_bin/MF_fake.jsonl, length:97
  0% 0/97 [00:00<?, ?it/s][transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
0 57
58 122
124 202
206 280
282 306
310 350
352 436
438 484
486 538
542 578
580 648
  1% 1/97 [00:00<00:33,  2.85it/s]0 57
58 122
124 158
161 197
199 247
250 294
296 328
331 421
0 57
98 128
131 173
17

### Step 4A-gpt2 — released CRF/BMES head on GPT-2 features

In [18]:
!python LIFE_train/train.py --split_dataset --data_path "{FEATURES_GPT2_BIN}" --train_path "{TRAIN_PATH_GPT2}" --test_path "{TEST_PATH_GPT2}" --model Transformer --num_train_epochs 50

Log INFO: split dataset...
********************************
The overall data sources:
['MF_fake.jsonl', 'MR_true.jsonl']
100% 183/183 [00:00<00:00, 4037.38it/s]
100% 46/46 [00:00<00:00, 3618.36it/s]

The number of train dataset: 183
The number of test  dataset: 46
********************************
100% 183/183 [00:00<00:00, 775.52it/s]
100% 46/46 [00:00<00:00, 7657.79it/s]
--------------------------------classify--------------------------------
Log INFO: do train...
Epoch:   0% 0/50 [00:00<?, ?it/s]
Iteration:   0% 0/6 [00:00<?, ?it/s]
Iteration:  17% 1/6 [00:00<00:02,  1.69it/s]
Iteration:  33% 2/6 [00:00<00:01,  3.17it/s]
Iteration:  50% 3/6 [00:00<00:00,  4.37it/s]
Iteration:  67% 4/6 [00:00<00:00,  5.41it/s]
Iteration: 100% 6/6 [00:01<00:00,  5.20it/s]
epoch 1: train_loss 2.7114956776301065

Iteration:   0% 0/2 [00:00<?, ?it/s]
Iteration:  50% 1/2 [00:00<00:00,  1.96it/s]
Iteration: 100% 2/2 [00:00<00:00,  2.30it/s]
******** Evalation ********
Accuracy: 51.7
Macro F1 Score: 45.1
Pre

### Step 4B-gpt2 — paper BCE head on GPT-2 features
Same GPT-2 features and the same deterministic seed-0 split as 4A. To match the LLaMA BCE
headline, re-run with `--seed 1..20` and report mean±std (~46-article test set ≈ 2 pts/article).

In [47]:
!python LIFE_train/train_bce.py \
  --split_dataset \
  --data_path "{FEATURES_GPT2_BIN}" \
  --train_path "{TRAIN_PATH_GPT2}" \
  --test_path "{TEST_PATH_GPT2}" \
  --num_train_epochs 50 \
  --seed 20

Log INFO: split dataset...
********************************
The overall data sources:
['MF_fake.jsonl', 'MR_true.jsonl']
100% 183/183 [00:00<00:00, 4158.40it/s]
100% 46/46 [00:00<00:00, 3776.06it/s]

The number of train dataset: 183
The number of test  dataset: 46
********************************
100% 183/183 [00:00<00:00, 7605.23it/s]
100% 46/46 [00:00<00:00, 7769.11it/s]
seed: 20
--------------------------------BCE head (paper Eq 11-12)--------------------------------
Log INFO: do train...
Epoch:   0% 0/50 [00:00<?, ?it/s]
Iteration:   0% 0/6 [00:00<?, ?it/s]
Iteration:  17% 1/6 [00:00<00:02,  1.75it/s]
Iteration:  33% 2/6 [00:00<00:01,  3.37it/s]
Iteration:  67% 4/6 [00:00<00:00,  5.79it/s]
Iteration: 100% 6/6 [00:01<00:00,  5.66it/s]
epoch 1: train_loss 0.7076544761657715

Iteration:   0% 0/2 [00:00<?, ?it/s]
Iteration: 100% 2/2 [00:00<00:00, 13.79it/s]
******** Evalation ********
Accuracy: 63.0
Macro F1 Score: 38.7
Precision/Recall per class (real=0, fake=1): 
63.0/100.0 0.0/0.0
*

## Notes / troubleshooting
- **HF gating**: Step 3 defaults to the ungated `NousResearch/Llama-2-7b-hf` mirror (no token). If you switch to the official `meta-llama` repo and hit a 403 "gated repo", your access request hasn't been approved yet.
- **fastNLP**: if Step 4 errors on `from fastNLP.modules.torch import ...`, run `!pip install -q fastNLP==1.0.1` and restart the runtime.
- **Checkpoints**: `BERT_CKPT` (step 1) and `linear_en.pt` (step 4) are written under `PROJECT_DIR` on Drive, so they survive disconnects.
- **NaN features**: if Step 3 prints NaN/inf, switch Step 3 to `--dtype float32` (fits the 40 GB A100).
- **Re-runs**: Step 2 mutates `OUTPUT_BIN` in place; always re-run Step 0 before re-running Steps 1–3 from scratch.